# Phase 1 Final Governance Reconciliation

This notebook is an orchestration and audit surface only. Durable scientific logic lives in `src/` and is exercised through `python -m src.cli phase1_final_governance_reconciliation`.

Scientific integrity boundary:

- This step links final comparator reconciliation to final controls, calibration, influence and reporting manifests.
- It does not run new models, alter logits, recompute efficacy metrics or fabricate missing governance artifacts.
- If final control/calibration/influence/reporting manifests are absent, the correct result is blocked/non-claim.
- Even if all governance manifests become present later, this runner remains claim-closed and records readiness for review; it does not itself open headline claims.
- Do not interpret comparator metrics as Phase 1 evidence until the full preregistered governance package is complete and reviewed.

In [ ]:
# Cell 1 - Bootstrap repo and run unit tests.

from __future__ import annotations

import getpass
import hashlib
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped or unavailable:', exc)

REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'
REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
RUN_UNITTESTS = True

def run(cmd, cwd=None, check=True):
    printable = ['<redacted>' if 'Authorization:' in str(part) else str(part) for part in cmd]
    print('$', ' '.join(printable))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True)

if not REPO_DIR.exists():
    token = getpass.getpass('Enter GitHub token for private repo, or leave blank for public clone: ')
    if token:
        header = 'Authorization: Basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()
        run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
    else:
        run(['git', 'clone', REPO_URL, str(REPO_DIR)])
else:
    run(['git', 'pull', '--ff-only'], cwd=REPO_DIR)

print('Repo:', REPO_DIR)
run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)

if RUN_UNITTESTS:
    unit = subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, text=True)
    if unit.returncode != 0:
        raise subprocess.CalledProcessError(unit.returncode, ['python', '-m', 'unittest', 'discover', '-s', 'tests'])

In [ ]:
# Cell 2 - Select reviewed source artifacts and keep launch disabled by default.

EXPECTED_PREREG_IDENTITY_HASH = '87e928ea747099c336a32121bc156655a1a160d666a251c7ac41228efba96af6'
PREREG_BUNDLE = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z/prereg_bundle.json'

COMPARATOR_RECONCILIATION_RUN = None  # Optional explicit Path(...); defaults to latest artifact.

# Optional future manifests. Leave as None until real final governance packages exist.
FINAL_CONTROL_MANIFEST = None
FINAL_CALIBRATION_MANIFEST = None
FINAL_INFLUENCE_MANIFEST = None
FINAL_REPORTING_MANIFEST = None

RUN_FINAL_GOVERNANCE_RECONCILIATION = False
REQUIRED_ACK = 'I understand this governance reconciliation is claim-closed and must not fabricate missing controls calibration influence or reporting results'
MANUAL_LAUNCH_ACK = ''

def latest_run(root: Path) -> Path:
    latest = root / 'latest.txt'
    if latest.exists():
        candidate = Path(latest.read_text().strip())
        if candidate.exists():
            return candidate
    runs = sorted([path for path in root.iterdir() if path.is_dir()])
    if not runs:
        raise FileNotFoundError(f'No runs under {root}')
    return runs[-1]

if COMPARATOR_RECONCILIATION_RUN is None:
    COMPARATOR_RECONCILIATION_RUN = latest_run(DRIVE_ROOT / 'artifacts/phase1_final_comparator_reconciliation')

PLAN_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_governance_reconciliation_plan'
OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase1_final_governance_reconciliation'

print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)
print('Final control manifest:', FINAL_CONTROL_MANIFEST)
print('Final calibration manifest:', FINAL_CALIBRATION_MANIFEST)
print('Final influence manifest:', FINAL_INFLUENCE_MANIFEST)
print('Final reporting manifest:', FINAL_REPORTING_MANIFEST)
print('Run flag:', RUN_FINAL_GOVERNANCE_RECONCILIATION)

In [ ]:
# Cell 3 - Validate prereg identity and comparator reconciliation boundary.

def read_json(path: Path):
    return json.loads(path.read_text())

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

bundle = read_json(PREREG_BUNDLE)
actual_prereg_hash = bundle.get('prereg_bundle_hash_sha256')
raw_prereg_sha256 = sha256(PREREG_BUNDLE)
print('Prereg status:', bundle.get('status'))
print('Locked prereg identity hash:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
assert bundle.get('status') == 'locked', 'Prereg bundle must be locked.'
assert actual_prereg_hash == EXPECTED_PREREG_IDENTITY_HASH, 'Locked prereg identity hash mismatch.'

comp_summary = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciliation_summary.json')
comp_claim_state = read_json(COMPARATOR_RECONCILIATION_RUN / 'phase1_final_comparator_reconciled_claim_state.json')

preflight_blockers = []
if comp_summary.get('status') != 'phase1_final_comparator_reconciliation_complete_claim_closed':
    preflight_blockers.append('comparator_reconciliation_not_complete_claim_closed')
if comp_summary.get('all_final_comparator_outputs_present') is not True:
    preflight_blockers.append('comparator_outputs_not_complete')
if comp_summary.get('runtime_comparator_logs_audited_for_all_required_comparators') is not True:
    preflight_blockers.append('runtime_logs_not_audited_for_all_comparators')
if comp_summary.get('claim_ready') is not False or comp_claim_state.get('claim_ready') is not False:
    preflight_blockers.append('comparator_reconciliation_claim_not_closed')

optional_manifests = {
    'final_control_manifest': FINAL_CONTROL_MANIFEST,
    'final_calibration_manifest': FINAL_CALIBRATION_MANIFEST,
    'final_influence_manifest': FINAL_INFLUENCE_MANIFEST,
    'final_reporting_manifest': FINAL_REPORTING_MANIFEST,
}
missing_optional = [name for name, value in optional_manifests.items() if value is None]

print('Comparator reconciliation status:', comp_summary.get('status'))
print('Completed comparators:', comp_summary.get('completed_comparators'))
print('Blocked comparators:', comp_summary.get('blocked_comparators'))
print('Comparator outputs complete:', comp_summary.get('all_final_comparator_outputs_present'))
print('Runtime logs audited:', comp_summary.get('runtime_comparator_logs_audited_for_all_required_comparators'))
print('Optional governance manifests missing:', missing_optional)
print('Preflight blockers:', preflight_blockers)

In [ ]:
# Cell 4 - Record governance reconciliation plan/manual hold artifact.

timestamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
plan_dir = PLAN_ROOT / timestamp
plan_dir.mkdir(parents=True, exist_ok=False)
plan = {
    'status': 'phase1_final_governance_reconciliation_plan_recorded',
    'created_utc': timestamp,
    'run_flag': RUN_FINAL_GOVERNANCE_RECONCILIATION,
    'required_ack': REQUIRED_ACK,
    'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
    'preflight_blockers': preflight_blockers,
    'expected_missing_until_final_governance_manifests_exist': missing_optional,
    'prereg_bundle': str(PREREG_BUNDLE),
    'locked_prereg_identity_hash': actual_prereg_hash,
    'raw_prereg_file_sha256': raw_prereg_sha256,
    'comparator_reconciliation_run': str(COMPARATOR_RECONCILIATION_RUN),
    'output_root': str(OUTPUT_ROOT),
    'scope': 'final governance reconciliation only; no fabricated controls/calibration/influence/reporting results',
    'not_ok_to_claim': [
        'decoder efficacy',
        'A2d efficacy',
        'A3/A4 efficacy',
        'A4 superiority',
        'privileged-transfer efficacy',
        'full Phase 1 neural comparator performance',
    ],
}
(plan_dir / 'phase1_final_governance_reconciliation_plan.json').write_text(json.dumps(plan, indent=2) + '\n')
print(json.dumps(plan, indent=2))

In [ ]:
# Cell 5 - Manual hold or launch governance reconciliation CLI.

if preflight_blockers:
    raise RuntimeError(f'Preflight blockers must be resolved before governance reconciliation: {preflight_blockers}')
elif not RUN_FINAL_GOVERNANCE_RECONCILIATION:
    hold = {
        'status': 'phase1_final_governance_reconciliation_manual_hold',
        'plan_dir': str(plan_dir),
        'run_flag': RUN_FINAL_GOVERNANCE_RECONCILIATION,
        'required_ack': REQUIRED_ACK,
        'ack_matched': MANUAL_LAUNCH_ACK == REQUIRED_ACK,
        'expected_result_if_launched_now': 'blocked_non_claim_if_final_governance_manifests_are_missing',
    }
    print(json.dumps(hold, indent=2))
elif MANUAL_LAUNCH_ACK != REQUIRED_ACK:
    raise RuntimeError('Manual launch acknowledgement mismatch. Do not reconcile governance without explicit non-fabrication acknowledgement.')
else:
    cmd = [
        'python', '-m', 'src.cli', 'phase1_final_governance_reconciliation',
        '--profile', 't4_safe',
        '--config', str(PREREG_BUNDLE),
        '--comparator-reconciliation-run', str(COMPARATOR_RECONCILIATION_RUN),
        '--output-root', str(OUTPUT_ROOT),
        '--governance-config', 'configs/phase1/final_governance_reconciliation.json',
    ]
    optional_args = [
        ('--final-control-manifest', FINAL_CONTROL_MANIFEST),
        ('--final-calibration-manifest', FINAL_CALIBRATION_MANIFEST),
        ('--final-influence-manifest', FINAL_INFLUENCE_MANIFEST),
        ('--final-reporting-manifest', FINAL_REPORTING_MANIFEST),
    ]
    for flag, value in optional_args:
        if value is not None:
            cmd.extend([flag, str(value)])
    run(cmd, cwd=REPO_DIR)
    print('Final governance reconciliation command completed. Review blockers before any downstream claim-state work.')

In [ ]:
# Cell 6 - Inspect latest governance reconciliation output if execution ran.

latest = OUTPUT_ROOT / 'latest.txt'
print('latest exists:', latest.exists())
if latest.exists():
    run_dir = Path(latest.read_text().strip())
    print('run_dir:', run_dir)
    required_files = [
        'phase1_final_governance_reconciliation_inputs.json',
        'phase1_final_governance_reconciliation_summary.json',
        'phase1_final_governance_reconciliation_report.md',
        'phase1_final_governance_reconciliation_source_links.json',
        'phase1_final_governance_reconciliation_input_validation.json',
        'phase1_final_controls_reconciliation_status.json',
        'phase1_final_calibration_reconciliation_status.json',
        'phase1_final_influence_reconciliation_status.json',
        'phase1_final_reporting_reconciliation_status.json',
        'phase1_final_governance_claim_state.json',
    ]
    for name in required_files:
        print(name, 'exists =', (run_dir / name).exists())
    summary = read_json(run_dir / 'phase1_final_governance_reconciliation_summary.json')
    controls = read_json(run_dir / 'phase1_final_controls_reconciliation_status.json')
    calibration = read_json(run_dir / 'phase1_final_calibration_reconciliation_status.json')
    influence = read_json(run_dir / 'phase1_final_influence_reconciliation_status.json')
    reporting = read_json(run_dir / 'phase1_final_reporting_reconciliation_status.json')
    claim_state = read_json(run_dir / 'phase1_final_governance_claim_state.json')
    print(json.dumps({
        'status': summary.get('status'),
        'comparator_outputs_complete': summary.get('comparator_outputs_complete'),
        'runtime_logs_audited_for_all_required_comparators': summary.get('runtime_logs_audited_for_all_required_comparators'),
        'governance_surfaces': summary.get('governance_surfaces'),
        'claim_ready': summary.get('claim_ready'),
        'claim_blockers': summary.get('claim_blockers'),
        'controls_status': controls.get('status'),
        'calibration_status': calibration.get('status'),
        'influence_status': influence.get('status'),
        'reporting_status': reporting.get('status'),
    }, indent=2))
else:
    print('No governance reconciliation run was launched in this pass.')

In [ ]:
# Cell 7 - Assertions for launched governance reconciliation output.

if RUN_FINAL_GOVERNANCE_RECONCILIATION:
    assert latest.exists(), 'Governance reconciliation latest pointer missing.'
    assert summary.get('status') in {
        'phase1_final_governance_reconciliation_blocked',
        'phase1_final_governance_reconciliation_ready_claim_closed',
    }, summary.get('claim_blockers')
    assert summary.get('comparator_outputs_complete') is True
    assert summary.get('runtime_logs_audited_for_all_required_comparators') is True
    assert summary.get('claim_ready') is False
    assert summary.get('headline_phase1_claim_open') is False
    assert summary.get('full_phase1_claim_bearing_run_allowed') is False
    if missing_optional:
        assert summary.get('status') == 'phase1_final_governance_reconciliation_blocked'
        assert 'controls_calibration_influence_reporting_missing' in summary.get('claim_blockers', [])
    print('Governance reconciliation assertions passed. Claims remain closed.')
else:
    print('Assertions skipped because manual flag is False.')

In [ ]:
# Cell 8 - Write a non-claim review note after launched governance reconciliation.

if RUN_FINAL_GOVERNANCE_RECONCILIATION and latest.exists():
    review_note = {
        'status': 'phase1_final_governance_reconciliation_review_pass_non_claim',
        'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
        'reviewed_run': str(run_dir),
        'scope': 'final controls/calibration/influence/reporting reconciliation only',
        'checks_passed': [
            'comparator_reconciliation_linked',
            'all_final_comparator_outputs_present_upstream',
            'all_runtime_comparator_logs_audited_upstream',
            'claim_ready_false',
            'headline_phase1_claim_open_false',
            'no_missing_governance_results_fabricated',
        ],
        'governance_surfaces': summary.get('governance_surfaces'),
        'claim_blockers': summary.get('claim_blockers'),
        'expected_current_interpretation': 'blocked_non_claim_until_final_controls_calibration_influence_reporting_manifests_exist_and_pass',
        'not_ok_to_claim': [
            'decoder efficacy',
            'A2d efficacy',
            'A3/A4 efficacy',
            'A4 superiority',
            'privileged-transfer efficacy',
            'full Phase 1 neural comparator performance',
        ],
    }
    review_path = run_dir / 'phase1_final_governance_reconciliation_review_note.json'
    review_path.write_text(json.dumps(review_note, indent=2) + '\n')
    print('Review note written:', review_path)
    print(json.dumps(review_note, indent=2))
else:
    print('Review note skipped because governance reconciliation was not launched.')

In [ ]:
# Cell 9 - Closeout.

print('================ Phase 1 Final Governance Reconciliation Closeout ================')
print('Notebook fix marker: phase1_final_governance_reconciliation_v1_20260422')
print('Plan source of truth:', plan_dir)
print('Run flag:', RUN_FINAL_GOVERNANCE_RECONCILIATION)
print('Comparator reconciliation run:', COMPARATOR_RECONCILIATION_RUN)
print('Expected locked prereg identity hash:', EXPECTED_PREREG_IDENTITY_HASH)
print('Locked prereg hash from bundle:', actual_prereg_hash)
print('Raw prereg file SHA256:', raw_prereg_sha256)
if RUN_FINAL_GOVERNANCE_RECONCILIATION and latest.exists():
    print('Latest governance reconciliation output:', run_dir)
    print('Comparator outputs complete:', summary.get('comparator_outputs_complete'))
    print('Runtime logs audited:', summary.get('runtime_logs_audited_for_all_required_comparators'))
    print('Governance surfaces:', summary.get('governance_surfaces'))
    print('Claim blockers:', summary.get('claim_blockers'))
    print('CHECK REQUIRED: Review controls/calibration/influence/reporting status before any final claim-state work.')
else:
    print('HELD: Governance reconciliation was not launched because manual flag is False.')
    print('NEXT: review the plan, then rerun only with explicit non-fabrication acknowledgement if appropriate.')
print('\nNOT OK TO CLAIM: This notebook does not prove decoder efficacy, A2d efficacy, A3/A4 efficacy, A4 superiority, privileged-transfer efficacy, or full Phase 1 neural comparator performance.')